# Portfolio End-to-End Demo

这个 notebook 演示一条完整的 portfolio workflow：

1. 生成一个小型 synthetic factor lake 与 mock kline
2. 运行 multiple_factor_composite 生成组合信号
3. 运行 holdings_gen 把信号转换为 holdings
4. 运行 portfolio_backtest 输出回测结果

这个版本是 self-contained demo，不依赖外部真实数据路径。

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'portfolio_demo' else Path.cwd().resolve()
if not (REPO_ROOT / 'demo' / 'portfolio_demo').exists():
    if (Path.cwd().resolve().parent / 'demo' / 'portfolio_demo').exists():
        REPO_ROOT = Path.cwd().resolve().parent

DEMO_ROOT = REPO_ROOT / 'demo' / 'portfolio_demo'
CONFIGS_ROOT = DEMO_ROOT / 'configs'
FACTOR_LAKE_ROOT = DEMO_ROOT / 'factor_lake'
INPUTS_ROOT = DEMO_ROOT / 'inputs'
OUTPUTS_ROOT = DEMO_ROOT / 'outputs'

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from backtest_layer.portfolio_backtest.config_runner import run_from_config as run_portfolio_backtest
from strategy_layer.portfolio_alpha.holdings_gen.pipeline import run_from_config as run_holdings_gen
from strategy_layer.portfolio_alpha.multiple_factor_composite.pipeline import run_from_config as run_composite_signal

print({'repo_root': str(REPO_ROOT), 'demo_root': str(DEMO_ROOT)})

{'repo_root': '/home/yluel/share/projects/quantsociety_backend_project', 'demo_root': '/home/yluel/share/projects/quantsociety_backend_project/demo/portfolio_demo'}


In [2]:
def _write_factor(lake_root: Path, factor_id: str, rows: list[tuple[str, str, float]]) -> None:
    frame = pd.DataFrame(rows, columns=['datetime', 'asset', 'value'])
    frame['datetime'] = pd.to_datetime(frame['datetime'])
    year = int(frame['datetime'].dt.year.iloc[0])
    target = lake_root / 'factors' / factor_id / f'year={year}'
    target.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(target / 'data.parquet', index=False)


def _build_demo_inputs(demo_root: Path) -> dict[str, str]:
    factor_lake_root = demo_root / 'factor_lake'
    inputs_root = demo_root / 'inputs'
    factor_lake_root.mkdir(parents=True, exist_ok=True)
    inputs_root.mkdir(parents=True, exist_ok=True)

    dates = pd.date_range('2024-01-02', periods=10, freq='D')
    symbols = ['AAA', 'BBB', 'CCC', 'DDD']

    value_base = {'AAA': 4.0, 'BBB': 3.0, 'CCC': 2.0, 'DDD': 1.0}
    quality_base = {'AAA': 3.6, 'BBB': 3.1, 'CCC': 1.8, 'DDD': 1.0}
    price_start = {'AAA': 100.0, 'BBB': 80.0, 'CCC': 60.0, 'DDD': 40.0}
    daily_return = {'AAA': 0.012, 'BBB': 0.008, 'CCC': -0.004, 'DDD': -0.007}

    value_rows: list[tuple[str, str, float]] = []
    quality_rows: list[tuple[str, str, float]] = []
    kline_rows: list[dict[str, object]] = []

    latest_close = dict(price_start)
    for day_idx, date in enumerate(dates):
        date_text = date.strftime('%Y-%m-%d')
        for symbol in symbols:
            value_rows.append((date_text, symbol, float(value_base[symbol] + 0.05 * day_idx)))
            quality_rows.append((date_text, symbol, float(quality_base[symbol] + (day_idx % 3) * 0.04)))

            if day_idx > 0:
                latest_close[symbol] = latest_close[symbol] * (1.0 + daily_return[symbol])
            close = round(float(latest_close[symbol]), 6)
            open_price = round(close / (1.0 + daily_return[symbol] / 2.0), 6)
            kline_rows.append({
                'trade_date': date,
                'symbol': symbol,
                'open': open_price,
                'high': round(max(open_price, close) * 1.002, 6),
                'low': round(min(open_price, close) * 0.998, 6),
                'close': close,
                'volume': 100000 + day_idx * 1000,
            })

    _write_factor(factor_lake_root, 'demo_value_factor_v1', value_rows)
    _write_factor(factor_lake_root, 'demo_quality_factor_v1', quality_rows)

    kline_frame = pd.DataFrame(kline_rows)
    kline_path = inputs_root / 'mock_kline.parquet'
    kline_frame.to_parquet(kline_path, index=False)

    return {
        'factor_lake_root': str(factor_lake_root),
        'kline_path': str(kline_path),
    }


prepared_inputs = _build_demo_inputs(DEMO_ROOT)
prepared_inputs

{'factor_lake_root': '/home/yluel/share/projects/quantsociety_backend_project/demo/portfolio_demo/factor_lake',
 'kline_path': '/home/yluel/share/projects/quantsociety_backend_project/demo/portfolio_demo/inputs/mock_kline.parquet'}

In [3]:
previous_cwd = Path.cwd()
try:
    os.chdir(CONFIGS_ROOT)
    composite_result = run_composite_signal('composite_signal.yaml')
finally:
    os.chdir(previous_cwd)

holdings_result = run_holdings_gen(CONFIGS_ROOT / 'holdings_from_signal.yaml')
backtest_result = run_portfolio_backtest(CONFIGS_ROOT / 'portfolio_backtest.yaml')

composite_signal_path = Path(composite_result['outputs']['signal'])
if not composite_signal_path.is_absolute():
    composite_signal_path = (CONFIGS_ROOT / composite_signal_path).resolve()

summary = {
    'prepared_inputs': prepared_inputs,
    'composite': {
        'signal_rows': int(len(composite_result['signal'])),
        'signal_path': str(composite_signal_path),
    },
    'holdings': {
        'rows': int(len(holdings_result['holdings'])),
        'path': holdings_result['outputs']['holdings'],
        'summary': holdings_result['summary'],
    },
    'backtest': {
        'output_dir': backtest_result['output_dir'],
        'summary': backtest_result['backtest']['summary_df'].iloc[0].to_dict(),
    },
}

print(json.dumps(summary, ensure_ascii=False, indent=2, default=str))

{
  "prepared_inputs": {
    "factor_lake_root": "/home/yluel/share/projects/quantsociety_backend_project/demo/portfolio_demo/factor_lake",
    "kline_path": "/home/yluel/share/projects/quantsociety_backend_project/demo/portfolio_demo/inputs/mock_kline.parquet"
  },
  "composite": {
    "signal_rows": 40,
    "signal_path": "/home/yluel/share/projects/quantsociety_backend_project/demo/portfolio_demo/outputs/composite_signal_run/signals/composite_signal.parquet"
  },
  "holdings": {
    "rows": 20,
    "path": "/home/yluel/share/projects/quantsociety_backend_project/demo/portfolio_demo/outputs/holdings_run/holdings/holdings.parquet",
    "summary": {
      "trade_days": 10,
      "holdings_rows": 20,
      "unique_symbols": 2,
      "positions_per_day_mean": 2.0,
      "positions_per_day_max": 2,
      "gross_exposure_mean": 1.0,
      "gross_exposure_max": 1.0,
      "net_exposure_mean": 1.0,
      "net_exposure_max": 1.0,
      "trade_date_min": "2024-01-02 00:00:00",
      "trade_dat

In [6]:
signal_df = pd.read_parquet(composite_signal_path)
holdings_df = pd.read_parquet(holdings_result['outputs']['holdings'])
summary_path = Path(backtest_result['output_dir']) / 'summary.csv'
returns_path = Path(backtest_result['output_dir']) / 'returns.csv'
summary_df = pd.read_csv(summary_path)
returns_df = pd.read_csv(returns_path)

print('Signal Preview')
display(signal_df.head(8))
print('Holdings Preview')
display(holdings_df.head(8))
print('Backtest Summary')
display(summary_df)
print('Returns Tail')
display(returns_df[['trade_date', 'gross_return', 'net_return', 'nav_net']].tail())

Signal Preview


,timestamp,symbol,composite_score,rank,selected_flag,side,signal_id,signal_version
0,2024-01-02,AAA,1.00,1.0,True,LONG,demo_portfolio_multi_factor,v1
1,2024-01-02,BBB,0.75,2.0,True,LONG,demo_portfolio_multi_factor,v1
2,2024-01-02,CCC,0.50,3.0,False,NONE,demo_portfolio_multi_factor,v1
3,2024-01-02,DDD,0.25,4.0,False,NONE,demo_portfolio_multi_factor,v1
4,2024-01-03,AAA,1.00,1.0,True,LONG,demo_portfolio_multi_factor,v1
5,2024-01-03,BBB,0.75,2.0,True,LONG,demo_portfolio_multi_factor,v1
6,2024-01-03,CCC,0.50,3.0,False,NONE,demo_portfolio_multi_factor,v1
7,2024-01-03,DDD,0.25,4.0,False,NONE,demo_portfolio_multi_factor,v1


Holdings Preview


,trade_date,symbol,weight
0,2024-01-02,AAA,0.5
1,2024-01-02,BBB,0.5
2,2024-01-03,AAA,0.5
3,2024-01-03,BBB,0.5
4,2024-01-04,AAA,0.5
5,2024-01-04,BBB,0.5
6,2024-01-05,AAA,0.5
7,2024-01-05,BBB,0.5


Backtest Summary


,strategy_name,trade_days,total_return,annual_return,annual_volatility,sharpe,sortino,calmar,max_drawdown,monthly_win_rate,avg_turnover,avg_holding_count,avg_gross_exposure,cost_drag,top5_day_pnl_contribution,effective_asset_return_ratio,avg_daily_asset_return_coverage
0,demo_portfolio_multi_factor,10.0,0.082321,6.341244,0.06656,30.098933,NaN,NaN,0.0,1.0,0.1,1.8,0.9,0.000536,0.628931,0.888889,0.888889


Returns Tail


,trade_date,gross_return,net_return,nav_net
5,2024-01-07,0.01,0.01,1.050490
6,2024-01-08,0.01,0.01,1.060995
7,2024-01-09,0.01,0.01,1.071605
8,2024-01-10,0.01,0.01,1.082321
9,2024-01-11,0.00,0.00,1.082321


In [7]:
holdings_profile = holdings_df.groupby('trade_date').agg(
    holdings_count=('symbol', 'nunique'),
    gross_exposure=('weight', lambda s: float(s.abs().sum())),
    net_exposure=('weight', 'sum'),
)

print('Daily Holdings Profile')
display(holdings_profile)

print('Net NAV Series')
display(returns_df[['trade_date', 'nav_net']])

Daily Holdings Profile


,holdings_count,gross_exposure,net_exposure
trade_date,,,
2024-01-02,2,1.0,1.0
2024-01-03,2,1.0,1.0
2024-01-04,2,1.0,1.0
2024-01-05,2,1.0,1.0
2024-01-06,2,1.0,1.0
2024-01-07,2,1.0,1.0
2024-01-08,2,1.0,1.0
2024-01-09,2,1.0,1.0
2024-01-10,2,1.0,1.0


Net NAV Series


,trade_date,nav_net
0,2024-01-02,1.000000
1,2024-01-03,1.009500
2,2024-01-04,1.019595
3,2024-01-05,1.029791
4,2024-01-06,1.040089
5,2024-01-07,1.050490
6,2024-01-08,1.060995
7,2024-01-09,1.071605
8,2024-01-10,1.082321
9,2024-01-11,1.082321
